# A reusable Outlines model for smolagents
24-679 | Week 10 follow-up | Constrained actions with custom tools

The agent chooses its next action during each native smolagents step. `OutlinesToolModel` is the reusable product: it derives an Outlines schema from the supplied tools and constrains each generated action. Pass the model to `OutlinesToolCallingAgent` or `OutlinesCodeAgent`. The Python tools produce observations.

The instructor runs the demo. The saved output shows the actual agent run.


## Setup and guarantee
Run top to bottom in a fresh Colab runtime. A T4 GPU is helpful; CPU is slower. The first run downloads public [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) weights. The local model needs no API key. Inputs and shipping rules are fictional.

The install cell uses the [smolagents-outlines package](https://github.com/cmccomb/smolagents-outlines), which pins the tested smolagents, Outlines, Transformers, and Accelerate versions. The adapter turns each tool's declared scalar inputs into a tool-specific JSON schema. Outlines constrains generation; the adapter validates the result again before smolagents receives it. Unsupported input types fail at setup.

The final answer is a free-form string by default. Pass `final_answers=[...]` to restrict it to choices, or `final_answer_schema={...}` for another JSON result shape. Repeated valid calls and any call order remain allowed. This guarantees call shape according to the declared tool schema, not that the tool succeeds or that the final decision is correct.

`OutlinesToolCallingAgent` receives native tool calls. `OutlinesCodeAgent` receives a complete code block with one known tool invocation; it does not generate general Python. The helpers stop on action errors and step limits.


In [ ]:
%pip install -q "git+https://github.com/cmccomb/smolagents-outlines.git@v0.1.0"


## 1. The fictional packaging case
A prototype device measures **180 × 120 × 40 mm** and weighs **0.42 kg**. Allow **10 mm clearance on every face**, requiring a box interior of **200 × 140 × 60 mm**. For the invented service, billed mass is the larger of actual packed mass and external volume in cm³ divided by 5,000, rounded **up** to 0.1 kg. Delivery costs `$6` per billed kg plus box price, with a `$6.50` budget.

A useful strategy is to check fit before mass or cost, but the Outlines constraint here does not enforce that order. The agent chooses its calls during the run.


In [ ]:
import json
import math

CASE = {
    "device_mm": (180, 120, 40),
    "device_mass_kg": 0.42,
    "clearance_each_face_mm": 10,
    "dimensional_divisor_cm3_per_kg": 5000,
    "rate_usd_per_kg": 6.00,
    "budget_usd": 6.50,
}
BOXES = {
    "Compact": {"inner_mm": (198, 145, 65), "outer_mm": (218, 165, 85),
                "box_mass_kg": 0.16, "padding_mass_kg": 0.08,
                "box_price_usd": 1.25},
    "Standard": {"inner_mm": (210, 150, 70), "outer_mm": (230, 170, 90),
                 "box_mass_kg": 0.22, "padding_mass_kg": 0.10,
                 "box_price_usd": 1.55},
}

def box_record(package: str) -> dict:
    if package not in BOXES:
        raise ValueError(f"Unsupported package: {package!r}")
    box = BOXES[package]
    values = [*CASE["device_mm"], CASE["device_mass_kg"],
              CASE["clearance_each_face_mm"],
              CASE["dimensional_divisor_cm3_per_kg"],
              CASE["rate_usd_per_kg"], CASE["budget_usd"],
              *box["inner_mm"], *box["outer_mm"],
              box["box_mass_kg"], box["padding_mass_kg"],
              box["box_price_usd"]]
    if not all(isinstance(x, (int, float)) and not isinstance(x, bool)
               and math.isfinite(x) and x > 0 for x in values):
        raise ValueError("Case quantities must be positive finite numbers")
    if any(outer <= inner for inner, outer in
           zip(box["inner_mm"], box["outer_mm"])):
        raise ValueError("Outer dimensions must exceed inner dimensions")
    return box

def fit_metrics(package: str) -> dict:
    box = box_record(package)
    required = tuple(x + 2 * CASE["clearance_each_face_mm"]
                     for x in CASE["device_mm"])
    gaps = tuple(inner - need for inner, need in zip(box["inner_mm"], required))
    return {"package": package, "required_inner_mm": required,
            "gaps_mm": gaps, "fits": all(x >= 0 for x in gaps)}

def mass_metrics(package: str) -> dict:
    box = box_record(package)
    actual = (CASE["device_mass_kg"] + box["box_mass_kg"]
              + box["padding_mass_kg"])
    dimensional = (math.prod(box["outer_mm"]) / 1000
                   / CASE["dimensional_divisor_cm3_per_kg"])
    billed = math.ceil(max(actual, dimensional) * 10 - 1e-12) / 10
    return {"package": package, "actual_mass_kg": round(actual, 3),
            "dimensional_mass_kg": round(dimensional, 3),
            "billed_mass_kg": round(billed, 1)}

def cost_metrics(package: str) -> dict:
    box = box_record(package)
    billed = mass_metrics(package)["billed_mass_kg"]
    cost = round(box["box_price_usd"] + CASE["rate_usd_per_kg"] * billed, 2)
    return {"package": package, "box_price_usd": box["box_price_usd"],
            "delivered_cost_usd": cost,
            "within_budget": cost <= CASE["budget_usd"]}


## 2. Define the custom tools
Each tool validates the package through the fictional calculator and returns a compact JSON observation. There is no separate execution ledger or model-generated observation.


In [ ]:
from smolagents import Tool

CALCULATORS = {
    "check_fit": fit_metrics,
    "calculate_billed_mass": mass_metrics,
    "calculate_delivery_cost": cost_metrics,
}

class PackageTool(Tool):
    output_type = "string"

    def __init__(self, name, description):
        self.name = name
        self.description = description
        self.inputs = {"package": {
            "type": "string", "description": "Exactly Compact or Standard",
            "enum": list(BOXES),
        }}
        super().__init__()

    def forward(self, package: str) -> str:
        result = CALCULATORS[self.name](package)
        return json.dumps(result, sort_keys=True, allow_nan=False)

CUSTOM_TOOLS = [
    PackageTool("check_fit", "Check the box interior against device plus padding"),
    PackageTool("calculate_billed_mass", "Calculate packed and billable mass"),
    PackageTool("calculate_delivery_cost", "Calculate delivered cost and budget screen"),
]

# Direct calculator checks are separate from the agent run.
assert fit_metrics("Compact")["gaps_mm"] == (-2, 5, 5)
assert mass_metrics("Standard")["billed_mass_kg"] == 0.8
assert cost_metrics("Standard")["delivered_cost_usd"] == 6.35


## 3. Use the package model
The [package source](https://github.com/cmccomb/smolagents-outlines/blob/v0.1.0/src/smolagents_outlines/_core.py) contains the Outlines schema construction and `Model.generate()` adapter. It reads tool names and scalar input schemas from the supplied smolagents tools. The model returns one schema-valid action or raises an error.

The model is separate from the agent. The two small agent helpers handle native output format, the CodeAgent prompt, and explicit failure at a step limit.


In [ ]:
from smolagents_outlines import (
    OutlinesToolModel,
    OutlinesToolCallingAgent,
    OutlinesCodeAgent,
)


## 4. Pass the model to an agent
The final answer can vary because `OutlinesToolModel` defaults to a free-form string answer. Add `final_answers=[*BOXES, "none"]` when you want only those three results. The tool-call schema stays strict either way.

Switch `AGENT_CLASS` to `OutlinesCodeAgent` to run the same model through its single-call Python execution path. This demo uses native tool calls.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE in ("cuda", "mps") else torch.float32

# Public weights: avoid Colab's implicit HF_TOKEN secret prompt.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=False)
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map=DEVICE, dtype=DTYPE, token=False
)

model = OutlinesToolModel(hf_model, tokenizer, CUSTOM_TOOLS)
AGENT_CLASS = OutlinesToolCallingAgent  # Change to OutlinesCodeAgent if desired.
agent = AGENT_CLASS(
    tools=CUSTOM_TOOLS, model=model, max_steps=8,
    instructions=(
        "Compare the two fictional boxes with the available tools. "
        "Use tool observations to inform your final choice."
    ),
)


In [ ]:
agent_result = agent.run(
    "Choose the lowest delivered-cost box that fits the prototype "
    "with padding and stays within the fictional $6.50 budget."
)
print("Selected box:", agent_result)
